# 03 - Supervised fault detection with a Random Forest

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AHichem/industrial-anomaly-detection/blob/main/notebooks/03_supervised_random_forest.ipynb)

**Goal of this notebook**

Notebook 02 showed that the golden batch raises too many false alarms, and that the Isolation Forest barely detects the faults because it judges each point alone. Here we try a **supervised** model: we show it examples of faults and let it learn what they look like.

1. Cut every batch into **time windows** and describe each window with a few numbers (features).
2. Split the data **by batch** into train and test.
3. Train a **Random Forest** and measure false alarms and detection.
4. Look at **which variables** the model uses.
5. Test its main limit: a fault type it has **never seen**.

**Prerequisite:** run notebook 01 once, so that the clean data is cached in Google Drive.

## 1. Loading

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/indpensim')
except ImportError:  # not running in Colab
    DATA_DIR = Path('data')

PROCESS_PARQUET = DATA_DIR / 'indpensim_process.parquet'
assert PROCESS_PARQUET.exists(), 'Run notebook 01 first to create the cached data.'

df = pd.read_parquet(PROCESS_PARQUET)
df['t'] = df['Time (h)'].astype('float64').round(1)  # clean time key, identical across batches
print(df.shape, '-', df['batch_id'].nunique(), 'batches')

Mounted at /content/drive
(113935, 41) - 100 batches


Features naming simplification

In [2]:
INPUTS = {  # Inputs of the controlled system
    'Aeration rate(Fg:L/h)': 'aeration',
    'Agitator RPM(RPM:RPM)': 'agitator_rpm',
    'Sugar feed rate(Fs:L/h)': 'sugar_feed',
    'Acid flow rate(Fa:L/h)': 'acid_flow',
    'Base flow rate(Fb:L/h)': 'base_flow',
    'Heating/cooling water flow rate(Fc:L/h)': 'cooling_water',
    'Heating water flow rate(Fh:L/h)': 'heating_water',
    'Water for injection/dilution(Fw:L/h)': 'dilution_water',
    'Air head pressure(pressure:bar)': 'head_pressure',
    'Dumped broth flow(Fremoved:L/h)': 'dumped_broth',
    'PAA flow(Fpaa:PAA flow (L/h))': 'paa_flow',
    'Oil flow(Foil:L/hr)': 'oil_flow',
    'Ammonia shots(NH3_shots:kgs)': 'nh3_shots',
}
SENSORS = {  # measurements done online
    'Dissolved oxygen concentration(DO2:mg/L)': 'DO2',
    'pH(pH:pH)': 'pH',
    'Temperature(T:K)': 'temperature',
    'carbon dioxide percent in off-gas(CO2outgas:%)': 'CO2_offgas',
    'Oxygen in percent in off-gas(O2:O2  (%))': 'O2_offgas',
    'Vessel Weight(Wt:Kg)': 'weight',
}
COMPUTED = {  # derived from measurements
    'Vessel Volume(V:L)': 'volume',
    'Generated heat(Q:kJ)': 'heat',
    'Oxygen Uptake Rate(OUR:(g min^{-1}))': 'OUR',
    'Carbon evolution rate(CER:g/h)': 'CER',
}
OTHER = {
    'Penicillin concentration(P:g/L)': 'penicillin_true',     # target
    'Fault reference(Fault_ref:Fault ref)': 'fault',          # label
}
rename = {**INPUTS, **SENSORS, **COMPUTED, **OTHER}
missing_cols = [c for c in rename if c not in df.columns]
assert not missing_cols, f'Unexpected column names: {missing_cols}'
df = df.rename(columns=rename)

MONITOR_VARS = list(INPUTS.values()) + list(SENSORS.values()) + list(COMPUTED.values())
print(len(MONITOR_VARS), 'variables available to the monitor')

23 variables available to the monitor


## 2. Time windows

A model needs **one row per sample**. A single time point says little, so we cut every batch into **windows of 10 h** (0-10 h, 10-20 h...). Each window becomes one row, described by:

- the **mean** and **standard deviation** of each variable over the window;
- the window **position** in the batch (`window`), so the model knows the time - what the Isolation Forest was missing.

The label of a window is **1 if the fault is active at any moment in the window**, 0 otherwise.

In [ ]:
WINDOW_H = 10   # window length in hours

df['window'] = (df['t'] // WINDOW_H).astype(int)   # 0 = 0-10 h, 1 = 10-20 h, ...

windows = df.groupby(['batch_id', 'window'])[MONITOR_VARS].agg(['mean', 'std'])
windows.columns = ['_'.join(c) for c in windows.columns]   # ('DO2', 'mean') -> 'DO2_mean'
windows = windows.fillna(0)

windows['fault'] = df.groupby(['batch_id', 'window'])['fault'].max()   # 1 if the fault is active in the window
windows = windows.reset_index()

print('Windows:', len(windows), '- faulty windows:', int(windows['fault'].sum()))
windows.head()

Only a small share of the windows is faulty: the classes are **imbalanced**. We tell the model about it below with `class_weight='balanced'`.

## 3. Train / test split by batch

We split the **batches**, not the windows. If windows of the same batch were in both train and test, the model would recognise the batch rather than the fault (data leakage), and the test would look better than reality.

`stratify` keeps the same share of each group in train and test, so the test contains 3 of the 10 faulty batches.

In [ ]:
from sklearn.model_selection import train_test_split

batch_group = df.groupby('batch_id')['group'].first()
all_ids = list(range(1, 101))
fault_ids = list(range(91, 101))

train_ids, test_ids = train_test_split(all_ids, test_size=0.3, stratify=batch_group, random_state=0)
fault_test_ids = [b for b in test_ids if b in fault_ids]
print('Faulty batches in test:', fault_test_ids)

features = windows.columns.drop(['batch_id', 'fault'])   # 'window' stays: it gives the time
train = windows[windows['batch_id'].isin(train_ids)]
test  = windows[windows['batch_id'].isin(test_ids)]

X_train, y_train = train[features], train['fault']
X_test,  y_test  = test[features],  test['fault']
print(len(features), 'features,', len(X_train), 'train windows,', len(X_test), 'test windows')

## 4. Random Forest

A Random Forest is a set of decision trees, each trained on a random part of the data; the forest votes. It needs no scaling of the features and handles many features well.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=0)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

## 5. Results

Two standard metrics:

- **Precision:** when the model raises an alarm, how often is it a real fault?
- **Recall:** of all the fault windows, how many are detected?

We also give the false alarm rate and the detection rate, the same way as in notebook 02, to compare.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

print(confusion_matrix(y_test, y_pred))   # rows = true (0, 1), columns = predicted (0, 1)
print('Precision:', round(precision_score(y_test, y_pred), 2))
print('Recall:   ', round(recall_score(y_test, y_pred), 2))

In [ ]:
test = test.copy()
test['pred'] = y_pred

print('False alarms:', round(test.loc[test['fault'] == 0, 'pred'].mean() * 100, 1), '% of normal windows')
print('Detection:   ', round(test.loc[test['fault'] == 1, 'pred'].mean() * 100, 1), '% of fault windows')

**When does the model raise an alarm?** For each faulty test batch, the true fault and the prediction over time.

In [ ]:
for b in fault_test_ids:
    one = test[test['batch_id'] == b]
    plt.figure(figsize=(10, 2.5))
    plt.step(one['window'] * WINDOW_H, one['fault'], label='true fault')
    plt.step(one['window'] * WINDOW_H, one['pred'], label='prediction')
    plt.title(f'Batch {b}')
    plt.xlabel('Time (h)')
    plt.legend()
    plt.show()

**Your turn - interpretation** *(write 2-3 sentences here)*

- How many false alarms compared with the golden batch (15.5 %) and the Isolation Forest (1 %)?
- How many fault windows are detected compared with the Isolation Forest (2 %)?
- On the plots: is the fault detected at its start, late, or only partly?

## 6. Which variables does the model use?

The Random Forest tells how much each feature helped its decisions (feature importance).

In [ ]:
importance = pd.Series(model.feature_importances_, index=features).sort_values()
importance.tail(15).plot.barh(figsize=(7, 5), title='Top 15 features')
plt.show()

**Your turn - interpretation** *(write 1-2 sentences here)*

- Which variables come first? Do you find temperature and CO2 in the off-gas, where notebook 02 saw the faults most clearly?

## 7. The main limit: a fault type never seen

Notebook 01 showed that the faults come in **families** with the same timing (92/98, 93/96, 94/99/100, 91/97). With a random split, a faulty test batch usually has a "twin" in the training set, which makes the test easy.

Here we remove one family (93 and 96) from training and test the model on it: this is what happens in a plant when a **new kind of fault** appears.

In [ ]:
unseen = [93, 96]   # one fault family, removed from training

train2 = train[~train['batch_id'].isin(unseen)]   # same training set as before, without this family
model2 = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=0)
model2.fit(train2[features], train2['fault'])

new = windows[windows['batch_id'].isin(unseen)].copy()
new['pred'] = model2.predict(new[features])
print('Detection on a fault type never seen:', round(new.loc[new['fault'] == 1, 'pred'].mean() * 100, 1), '% of fault windows')

**Your turn - interpretation** *(write 1-2 sentences here)*

- Is the detection much lower than in section 5? What does it mean for a plant where new faults can appear?

## 8. Conclusions

*(Hichem - 3 to 4 bullet points: how the Random Forest compares with notebook 02, which variables matter, its limit on unseen faults, and why this motivates unsupervised models trained on normal batches only (notebook 04).)*